<a href="https://colab.research.google.com/github/Siya-Tambe/job-market-skill-gap/blob/main/job_market_skill_gap_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# These are external tools we need to borrow
# requests  → lets Python open a webpage like a browser
# beautifulsoup4 → lets Python read and pick parts of a webpage
# pandas → like Excel, but in Python
# matplotlib → draws charts

!pip install requests beautifulsoup4 pandas matplotlib

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt

print("All libraries loaded successfully")

All libraries loaded successfully


In [ ]:
# Step 1: Disguise Python as a browser
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Step 2: Define the URL first (as a plain string)
url = "https://internshala.com/internships/web-development-internship/"

# Step 3: NOW use url inside requests.get()
response = requests.get(url, headers=headers)

# Step 4: Check results
print("Status code:", response.status_code)
print("First 500 characters of HTML:")
print(response.text[:500])

Status code: 200
First 500 characters of HTML:
<!DOCTYPE html>
    <html xmlns="http://www.w3.org/1999/xhtml" xmlns:og="http://ogp.me/ns#" xmlns:fb="https://www.facebook.com/2008/fbml"
        lang="en-US">

<head>
    <meta http-equiv="X-UA-Compatible" content="IE=9" />
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0 user-scalable=0" />
    <meta property="fb:app_id" content="702141670710132" />
    <meta property="og:type" content="website" />
            <meta property="og:image:width"


In [ ]:
# Step 1 : Feed the HTML to BeautifulSoup
# 'html.parser' tells it what kind of document this is
soup = BeautifulSoup(response.text, 'html.parser')

# Step 2 : Find all job cards on site
# Each job listing is inside a div with this class name
job_cards = soup.find_all('div',class_='individual_internship')

# Step 3: See how many jobs we found
print("Number of jobs found:",len(job_cards))

# Step 4: Look at the raw HTML of just the first job card
# This helps us understand the structure before extracting
if len(job_cards) > 0:
  print("\nFirst Job card HTML")
  print(job_cards[0].prettify()[:1000]) # first 1000 characters only
else:
  print("No job cards found - the class name might have changed")

Number of jobs found: 51

First Job card HTML
<div class="container-fluid individual_internship view_detail_button visibilityTrackerItem" data-href="/internship/detail/backend-development-internship-in-mumbai-at-ace-3601775033759" data-source_page="search_page" employment_type="internship" id="individual_internship_3101435" internshipid="3101435" sequential_apply_referral="similar_internships">
 <div class="internship_meta duration_meta">
  <div class="internship-heading-container">
   <div class="company">
    <h2 class="job-internship-name">
     <a class="job-title-href" href="/internship/detail/backend-development-internship-in-mumbai-at-ace-3601775033759" id="job_title" target="_blank">
      Backend Development
     </a>
    </h2>
    <div class="heading_6 company_name">
     <div class="company_and_premium">
      <p class="company-name">
       Ace 360
      </p>
      <div class="actively-hiring-badge">
       Actively hiring
      </div>
     </div>
    </div>
   </div>
   <d

In [ ]:
# We'll store all jobs as a list of dictionaries
# A dictionary is like a row in Excel: {"column": "value"}

jobs = []  # empty list, we'll fill it up

# Loop through every job card we found
for card in job_cards:

    # --- Extract Job Title ---
    # Find the <a> tag with class 'job-title-href'

    title_tag = card.find('a', class_='job-title-href')
    title = title_tag.text.strip() if title_tag else "Not found"

    # .text = get the text inside the tag
    # .strip() = remove extra spaces and newlines
    # "if title_tag else" = safety net if tag doesn't exist

    # --- Extract Company Name ---
    company_tag = card.find('p', class_='company-name')
    company = company_tag.text.strip() if company_tag else "Not found"

    # --- Extract Skills ---
    # Skills are usually in <div class="skill-container"> or similar
    # Let's try to find them
    skills_tags = card.find_all('span', class_='round_tabs')

    if skills_tags:
        # If found, join all skill texts with a comma
        skills = ", ".join([s.text.strip() for s in skills_tags])
    else:
        skills = "Not listed"

    # --- Save this job as one dictionary ---
    job = {
        "title": title,
        "company": company,
        "skills": skills
    }

    jobs.append(job)  # add this job to our list

# Convert list of dictionaries → pandas DataFrame (like an Excel table)
df = pd.DataFrame(jobs)

# Show the first 5 rows
print(f"Total jobs collected: {len(df)}")
df.head()

Total jobs collected: 51


,title,company,skills
0,Backend Development,Ace 360,Not listed
1,PHP Development,InstaWeb Labs Private Limited,Not listed
2,Web Development,ATS Studio,Not listed
3,Front End Development,Arhant Solutions,Not listed
4,WordPress Development,ViralChilly,Not listed


In [ ]:
# We'll scrape 5 different job categories
# More data = more meaningful analysis

urls = {
    "Web Development":     "https://internshala.com/internships/web-development-internship/",
    "Data Science":        "https://internshala.com/internships/data-science-internship/",
    "Python":              "https://internshala.com/internships/python-internship/",
    "Machine Learning":    "https://internshala.com/internships/machine-learning-internship/",
    "Android Development": "https://internshala.com/internships/android-app-development-internship/"
}

all_jobs = []  # will store jobs from ALL categories

for category, url in urls.items():
  print(f"Scraping: {category}...")

  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, 'html.parser')
  job_cards = soup.find_all('div',class_='individual_internship')

  for card in job_cards:
    title_tag = card.find('a',class_='job-title-href')
    title = title_tag.text.strip() if title_tag else "Not found"

    company_tag = card.find('p',class_='company-name')
    company = company_tag.text.strip() if company_tag else "Not found"

    all_jobs.append({
        "title": title,
        "company": company,
        "category": category # <- new column! which domain is this job
    })

    print(f"Found {len(job_cards)} jobs")

# Convert to Dataframe
df = pd.DataFrame(all_jobs)
print(f"\nTotal jobs collected: {len(df)}")
df.head(10)

Scraping: Web Development...
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Scraping: Data Science...
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 51 jobs
Found 5

,title,company,category
0,Tech News Editor (WordPress),Abhinava Innovations,Web Development
1,Full Stack Development,Gharpayy,Web Development
2,Backend Development,Arhant Solutions,Web Development
3,Front End Development,Arhant Solutions,Web Development
4,Web Development,ATS Studio,Web Development
5,WordPress Development,ViralChilly,Web Development
6,Full Stack Development,Finflock Systems Private Limited,Web Development
7,Web Development,Eiosys Private Limited,Web Development
8,PHP Development,InstaWeb Labs Private Limited,Web Development
9,Machine Learning And Web Development,Greenleap Robotics Private Limited,Web Development


In [ ]:
# This is our master skill list
# We check if each skill appears in the job TITLE
# Real analysts call this "keyword matching"

skill_keywords = [
    # Languages
    "Python", "Java", "JavaScript", "PHP", "C++", "R", "Swift", "Kotlin",
    # Web
    "HTML", "CSS", "React", "Angular", "Vue", "Node", "Django", "Flask",
    "WordPress", "Frontend", "Backend", "Full Stack",
    # Data
    "SQL", "Excel", "Tableau", "Power BI", "Pandas", "NumPy",
    # ML/AI
    "Machine Learning", "Deep Learning", "NLP", "TensorFlow", "Computer Vision",
    # Mobile
    "Android", "iOS", "Flutter",
    # Other
    "Git", "API", "REST", "Firebase", "MongoDB"
]

# For each job, check which skills appear in its title
def find_skills(title):
  found = []
  title_lower = title.lower() # lowercase for fair comparison

  for skill in skill_keywords:
    # Check for whole word match, not partial
    # e.g. "node" should not match "knowledge"
    import re
    pattern = r'\b' + re.escape(skill.lower()) + r'\b'
    if re.search(pattern, title_lower):
            found.append(skill)
    return ", ".join(found) if found else "General"
    if skill.lower() in title_lower:
      found.append(skill)
  return ", ".join(found) if found else "General"

# apply this function to every row in our dataframe
#.apply() = run this function on each value in the column
df["skills_detected"] = df["title"].apply(find_skills)

print("Skills detected!")
df[["title","category","skills_detected"]].head(10)

Skills detected!


,title,category,skills_detected
0,Tech News Editor (WordPress),Web Development,General
1,Full Stack Development,Web Development,General
2,Backend Development,Web Development,General
3,Front End Development,Web Development,General
4,Web Development,Web Development,General
5,WordPress Development,Web Development,General
6,Full Stack Development,Web Development,General
7,Web Development,Web Development,General
8,PHP Development,Web Development,General
9,Machine Learning And Web Development,Web Development,General


In [ ]:
# Debug cell - let's test our function on ONE title manually
import re

test_title = "Full Stack Development"
test_title_lower = test_title.lower()

print(f"Testing title: '{test_title}'")
print(f"Lowercase: '{test_title_lower}'")
print()

# Test each skill manually
test_skills = ["Full Stack", "PHP", "WordPress", "Python", "Backend"]

for skill in test_skills:
    pattern = r'\b' + re.escape(skill.lower()) + r'\b'
    match = re.search(pattern, test_title_lower)
    print(f"Skill: '{skill}' | Pattern: '{pattern}' | Match: {match}")

Testing title: 'Full Stack Development'
Lowercase: 'full stack development'

Skill: 'Full Stack' | Pattern: '\bfull\ stack\b' | Match: <re.Match object; span=(0, 10), match='full stack'>
Skill: 'PHP' | Pattern: '\bphp\b' | Match: None
Skill: 'WordPress' | Pattern: '\bwordpress\b' | Match: None
Skill: 'Python' | Pattern: '\bpython\b' | Match: None
Skill: 'Backend' | Pattern: '\bbackend\b' | Match: None


In [21]:
from collections import Counter
import re

# ── STEP 1: Clear everything and start fresh ──────────────────────
all_jobs = []  # fresh empty list

urls = {
    "Web Development":     "https://internshala.com/internships/web-development-internship/",
    "Data Science":        "https://internshala.com/internships/data-science-internship/",
    "Python":              "https://internshala.com/internships/python-internship/",
    "Machine Learning":    "https://internshala.com/internships/machine-learning-internship/",
    "Android Development": "https://internshala.com/internships/android-app-development-internship/"
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

for category, url in urls.items():
    print(f"Scraping: {category}...", end=" ")
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    job_cards = soup.find_all('div', class_='individual_internship')

    for card in job_cards:
        title_tag = card.find('a', class_='job-title-href')
        title = title_tag.text.strip() if title_tag else "Not found"

        company_tag = card.find('p', class_='company-name')
        company = company_tag.text.strip() if company_tag else "Not found"

        all_jobs.append({
            "title": title,
            "company": company,
            "category": category
        })
    print(f"{len(job_cards)} jobs found")

# ── STEP 2: Build dataframe and remove duplicates ─────────────────
df = pd.DataFrame(all_jobs)
before = len(df)
df = df.drop_duplicates()   # remove exact duplicate rows
after = len(df)
print(f"\nTotal jobs: {before} → after removing duplicates: {after}")

# ── STEP 3: Apply skill detection ─────────────────────────────────
skill_keywords = [
    "Python", "Java", "JavaScript", "PHP", "Swift", "Kotlin",
    "HTML", "CSS", "React", "Angular", "Vue", "Node",
    "Django", "Flask", "WordPress",
    "Frontend", "Front End", "Backend", "Back End", "Full Stack",
    "SQL", "Excel", "Tableau", "Power BI", "Pandas",
    "Machine Learning", "Deep Learning", "NLP", "TensorFlow",
    "Android", "iOS", "Flutter", "API", "Firebase", "MongoDB"
]

def find_skills(title):
    found = []
    title_lower = title.lower()
    for skill in skill_keywords:
        pattern = r'\b' + re.escape(skill.lower()) + r'\b'
        if re.search(pattern, title_lower):
            found.append(skill)
    return ", ".join(found) if found else "General"

df["skills_detected"] = df["title"].apply(find_skills)
print("\nSkill detection done!")
print(df["category"].value_counts())

Scraping: Web Development... 51 jobs found
Scraping: Data Science... 51 jobs found
Scraping: Python... 43 jobs found
Scraping: Machine Learning... 51 jobs found
Scraping: Android Development... 51 jobs found

Total jobs: 247 → after removing duplicates: 243

Skill detection done!
category
Web Development        51
Data Science           51
Android Development    51
Machine Learning       47
Python                 43
Name: count, dtype: int64


In [22]:
from collections import Counter
import re

# ── Remove duplicates ─────────────────────────────────────────────
df = df.drop_duplicates().reset_index(drop=True)
print(f"Clean dataset: {len(df)} jobs\n")

# ── QUESTION 1: Top skills per category ───────────────────────────
print("=" * 50)
print("Q1: TOP SKILLS PER CATEGORY")
print("=" * 50)

for category in df["category"].unique():
    category_df = df[df["category"] == category]
    cat_skills = []

    for skills_string in category_df["skills_detected"]:
        if skills_string != "General":
            for s in skills_string.split(","):
                cat_skills.append(s.strip())

    if cat_skills:
        top = Counter(cat_skills).most_common(3)
        top_str = ", ".join([f"{s}({c})" for s, c in top])
        print(f"\n► {category} ({len(category_df)} jobs)")
        print(f"  Top 3 skills: {top_str}")
    else:
        print(f"\n► {category}: no specific skills detected")

# ── QUESTION 2: Vagueness analysis ───────────────────────────────
print("\n" + "=" * 50)
print("Q2: HOW MANY JOB TITLES ARE VAGUE?")
print("=" * 50)

total          = len(df)
general_count  = len(df[df["skills_detected"] == "General"])
specific_count = total - general_count
general_pct    = round((general_count  / total) * 100, 1)
specific_pct   = round((specific_count / total) * 100, 1)

print(f"\n  Total jobs         : {total}")
print(f"  Skill-specific     : {specific_count} ({specific_pct}%)")
print(f"  Vague/General      : {general_count}  ({general_pct}%)")
print(f"\n  Insight: {general_pct}% of listings use generic titles.")
print(f"  Scraping descriptions would give much richer data.")

# ── QUESTION 3: Cross-category must-learn skills ──────────────────
print("\n" + "=" * 50)
print("Q3: SKILLS THAT APPEAR ACROSS MULTIPLE DOMAINS")
print("=" * 50)

skill_categories = {}

for _, row in df.iterrows():
    if row["skills_detected"] != "General":
        for s in row["skills_detected"].split(","):
            skill = s.strip()
            if skill not in skill_categories:
                skill_categories[skill] = set()
            skill_categories[skill].add(row["category"])

print(f"\n{'Skill':<22} {'In N categories':<18} {'Which ones'}")
print("-" * 70)

for skill, cats in sorted(skill_categories.items(),
                           key=lambda x: len(x[1]),
                           reverse=True):
    if len(cats) >= 2:
        print(f"{skill:<22} {len(cats):<18} {', '.join(cats)}")

print("\n✅ Phase 4 complete!")

Clean dataset: 243 jobs

Q1: TOP SKILLS PER CATEGORY

► Web Development (51 jobs)
  Top 3 skills: WordPress(7), Full Stack(5), PHP(4)

► Data Science (51 jobs)
  Top 3 skills: WordPress(1)

► Python (43 jobs)
  Top 3 skills: Python(2), Full Stack(2), Backend(1)

► Machine Learning (47 jobs)
  Top 3 skills: Machine Learning(7), Python(4), Full Stack(1)

► Android Development (51 jobs)
  Top 3 skills: Flutter(22), Android(8), iOS(1)

Q2: HOW MANY JOB TITLES ARE VAGUE?

  Total jobs         : 243
  Skill-specific     : 76 (31.3%)
  Vague/General      : 167  (68.7%)

  Insight: 68.7% of listings use generic titles.
  Scraping descriptions would give much richer data.

Q3: SKILLS THAT APPEAR ACROSS MULTIPLE DOMAINS

Skill                  In N categories    Which ones
----------------------------------------------------------------------
Full Stack             3                  Python, Machine Learning, Web Development
Python                 3                  Python, Machine Learning, Web